# Explore one Green Taxi file

This notebook inspects the January 2025 file. It does not change raw data or run the production pipeline. See the [design](../docs/adr/001-pipeline-design.md) for the agreed rules and the [data review](../docs/data-review.md) for the full three-file findings.

First, from the repository root, run `uv sync --locked` and `uv run --locked python -m green_taxi.download` to stage the files locally. Use the project's `.venv` as the notebook kernel. Docker volumes are separate from the host `data/` folder.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

root = Path.cwd()
if not (root / 'pyproject.toml').is_file():
    root = root.parent
source = root / 'data/raw/green_tripdata_2025-01.parquet'
if not source.is_file():
    raise FileNotFoundError(
        f'Taxi data not found: {source}\n'
        'On your Mac, open a terminal in the repository folder and run:\n'
        'uv sync --locked\n'
        'uv run --locked python -m green_taxi.download\n'
        'Then run this notebook cell again.'
    )
rides = pd.read_parquet(source)
print(f'Rows: {len(rides):,}')
rides.head()

## Which columns do we need?

We group by pickup date and sum `total_amount`. Check their types and missing values first.

In [ ]:
columns = ['lpep_pickup_datetime', 'total_amount']
pd.DataFrame({
    'type': rides[columns].dtypes.astype(str),
    'missing': rides[columns].isna().sum(),
})

## Dates and amounts

Look for values that need attention. The agreed reporting period is January–March, regardless of the source filename. Negative and zero amounts are kept by the pipeline; they are not automatically errors.

In [ ]:
pickup = pd.to_datetime(rides['lpep_pickup_datetime'], errors='coerce')
amount = pd.to_numeric(rides['total_amount'], errors='coerce')
outside = pickup.notna() & ((pickup < '2025-01-01') | (pickup >= '2025-04-01'))
pd.Series({
    'first pickup': pickup.min(),
    'last pickup': pickup.max(),
    'missing or invalid pickups': pickup.isna().sum(),
    'pickups outside Jan–Mar': outside.sum(),
    'lowest amount': amount.min(),
    'highest amount': amount.max(),
    'missing, invalid or non-finite amounts': (~np.isfinite(amount)).sum(),
    'negative amounts': (amount < 0).sum(),
    'zero amounts': (amount == 0).sum(),
}, name='January file')

## What does this tell us?

A file's name does not guarantee every pickup belongs to that month. The pipeline checks the pickup date against the reporting period and counts excluded records. It keeps valid zero and negative amounts. These checks describe one file; the [data review](../docs/data-review.md) covers the full three-month batch.